In [2]:
import os
import sys

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PATH"] = "/opt/spark/bin:" + os.environ["PATH"]
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("SPARK_HOME:", os.environ["SPARK_HOME"])

# 2. Khởi tạo SparkSession và định nghĩa biến 'spark'
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FraudDataAnalysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

# Ẩn bớt các log thông báo rác, chỉ hiện lỗi
spark.sparkContext.setLogLevel("ERROR")

print("Khởi tạo Spark thành công! Phiên bản:", spark.version)

Python: /home/lucas/projects/fraud-bigdata/.venv/bin/python
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
SPARK_HOME: /opt/spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 11:45:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/11 11:45:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Khởi tạo Spark thành công! Phiên bản: 4.1.2


In [3]:

HDFS_RAW_PATH = "hdfs://localhost:9000/user/lucas/projects/fraud_project/bronze/fraud_300k_raw_temporal.csv"

raw_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(HDFS_RAW_PATH)

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

raw_df.printSchema()
raw_df.show(5, truncate=False)

Rows: 300000
Columns: 16
root
 |-- Transaction ID: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Transaction Amount: double (nullable = true)
 |-- Transaction Date: timestamp (nullable = true)
 |-- Payment Method: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Customer Age: integer (nullable = true)
 |-- Customer Location: string (nullable = true)
 |-- Device Used: string (nullable = true)
 |-- IP Address: string (nullable = true)
 |-- Shipping Address: string (nullable = true)
 |-- Billing Address: string (nullable = true)
 |-- Is Fraudulent: integer (nullable = true)
 |-- Account Age Days: integer (nullable = true)
 |-- Transaction Hour: integer (nullable = true)

+------------------------------------+------------------------------------+------------------+-------------------+--------------+----------------+--------+------------+-----------------+-----------+---------------+------------

In [4]:
raw_df.createOrReplaceTempView("raw_transactions")

spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(`Is Fraudulent`) AS fraud_count,
    ROUND(AVG(`Is Fraudulent`) * 100, 4) AS fraud_rate_percent,
    MIN(`Transaction Date`) AS min_date,
    MAX(`Transaction Date`) AS max_date
FROM raw_transactions
""").show(truncate=False)

+----------+-----------+------------------+-------------------+-------------------+
|total_rows|fraud_count|fraud_rate_percent|min_date           |max_date           |
+----------+-----------+------------------+-------------------+-------------------+
|300000    |14994      |4.998             |2024-01-29 11:52:41|2024-02-17 12:34:30|
+----------+-----------+------------------+-------------------+-------------------+



In [5]:
spark.sql("""
SELECT
    `Transaction ID`,
    `Transaction Amount`,
    `Transaction Date`,
    `Payment Method`,
    `Product Category`,
    `Device Used`,
    `Account Age Days`,
    `Transaction Hour`,
    `Is Fraudulent`
FROM raw_transactions
LIMIT 10
""").show(truncate=False)

+------------------------------------+------------------+-------------------+--------------+----------------+-----------+----------------+----------------+-------------+
|Transaction ID                      |Transaction Amount|Transaction Date   |Payment Method|Product Category|Device Used|Account Age Days|Transaction Hour|Is Fraudulent|
+------------------------------------+------------------+-------------------+--------------+----------------+-----------+----------------+----------------+-------------+
|28ce569e-33bf-4f57-b4f1-267901f2e430|43.58             |2024-01-29 11:52:41|bank transfer |health & beauty |tablet     |216             |11              |0            |
|fa1d4bfe-1e77-423d-aa74-8480992d6f90|649.91            |2024-01-29 11:52:41|PayPal        |toys & games    |mobile     |91              |11              |0            |
|0cc7c25e-9a01-4081-9cee-3d2973228423|107.41            |2024-01-29 11:52:47|PayPal        |home & garden   |mobile     |314             |11          

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

TARGET_COL = "Is Fraudulent"
DATE_COL = "Transaction Date"

df_pd = raw_df.toPandas()
df_pd[DATE_COL] = pd.to_datetime(df_pd[DATE_COL], errors="coerce")
df_pd = df_pd.reset_index(drop=True)

print("Loaded pandas shape:", df_pd.shape)
print("Fraud rate:", round(df_pd[TARGET_COL].mean() * 100, 4), "%")
print("Date range:", df_pd[DATE_COL].min(), "→", df_pd[DATE_COL].max())

Loaded pandas shape: (300000, 16)
Fraud rate: 4.998 %
Date range: 2024-01-29 11:52:41 → 2024-02-17 12:34:30


In [7]:
idx = np.arange(len(df_pd))
y = df_pd[TARGET_COL]

train_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=42,
    stratify=y
)

raw_train = df_pd.iloc[train_idx].copy().reset_index(drop=True)
raw_test = df_pd.iloc[test_idx].copy().reset_index(drop=True)

split_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(raw_train),
        "frauds": int(raw_train[TARGET_COL].sum()),
        "fraud_rate": raw_train[TARGET_COL].mean(),
        "start_date": raw_train[DATE_COL].min(),
        "end_date": raw_train[DATE_COL].max()
    },
    {
        "split": "test",
        "rows": len(raw_test),
        "frauds": int(raw_test[TARGET_COL].sum()),
        "fraud_rate": raw_test[TARGET_COL].mean(),
        "start_date": raw_test[DATE_COL].min(),
        "end_date": raw_test[DATE_COL].max()
    }
])

display(split_summary)

,split,rows,frauds,fraud_rate,start_date,end_date
0,train,240000,11995,0.049979,2024-01-29 11:52:41,2024-02-17 12:34:30
1,test,60000,2999,0.049983,2024-01-29 11:52:54,2024-02-17 12:34:14


In [8]:
fraud_test = raw_test[raw_test[TARGET_COL] == 1]
nonfraud_test = raw_test[raw_test[TARGET_COL] == 0]

stream_fraud = fraud_test.sample(n=50, random_state=42)
stream_nonfraud = nonfraud_test.sample(n=950, random_state=42)

stream_demo = pd.concat([stream_fraud, stream_nonfraud], axis=0)

# Sort theo thời gian để mô phỏng dòng giao dịch phát sinh
stream_demo = stream_demo.sort_values(DATE_COL).reset_index(drop=True)

stream_summary = pd.DataFrame([{
    "file": "stream_demo_1000.csv",
    "rows": len(stream_demo),
    "frauds": int(stream_demo[TARGET_COL].sum()),
    "fraud_rate": stream_demo[TARGET_COL].mean(),
    "start_date": stream_demo[DATE_COL].min(),
    "end_date": stream_demo[DATE_COL].max()
}])

display(stream_summary)

stream_demo.head(10)

,file,rows,frauds,fraud_rate,start_date,end_date
0,stream_demo_1000.csv,1000,50,0.05,2024-01-29 12:01:33,2024-02-17 12:08:51


,Transaction ID,Customer ID,Transaction Amount,Transaction Date,Payment Method,Product Category,Quantity,Customer Age,Customer Location,Device Used,IP Address,Shipping Address,Billing Address,Is Fraudulent,Account Age Days,Transaction Hour
0,06f11434-ac9e-4ef2-9663-c1947b37f8d6,f9abaed3-56dc-4669-ba4d-9a212aaf99a0,699.50,2024-01-29 12:01:33,PayPal,clothing,4,18,Pittmanstad,desktop,176.178.181.125,"863 Karen Inlet Apt. 829\nMargaretfurt, IN 45425","863 Karen Inlet Apt. 829\nMargaretfurt, IN 45425",1,343,12
1,320449d0-51fc-4dee-8a79-f36683ca52c4,2c858748-a4a5-490e-b206-cc2785f6d384,134.36,2024-01-29 12:29:06,debit card,toys & games,2,39,South Laurashire,desktop,61.105.198.76,3126 Thomas Expressway Suite 316\nNorth Willia...,3126 Thomas Expressway Suite 316\nNorth Willia...,0,199,12
2,4b590d1a-73c0-4eed-9278-df32700ec333,18262686-b59f-4ec6-a060-5ce32d25e582,456.98,2024-01-29 12:51:03,credit card,health & beauty,3,39,East Jessicamouth,mobile,145.78.139.74,"513 Hutchinson Track Suite 824\nNew Laurie, WV...","513 Hutchinson Track Suite 824\nNew Laurie, WV...",0,141,12
3,3fb06b6f-4e42-462a-a577-c1884071929f,736d6568-dd5d-4d20-9930-09c93e4a4248,111.25,2024-01-29 13:01:49,debit card,home & garden,2,28,East Janet,desktop,192.161.98.175,"0064 Morales Avenue Suite 854\nKempbury, MP 77356","0064 Morales Avenue Suite 854\nKempbury, MP 77356",0,232,13
4,be8376e6-fa95-43a8-a922-cce11fefe19d,df6ac60b-6eea-497b-87f6-d8c5ff09802e,120.58,2024-01-29 14:42:59,credit card,electronics,3,44,Marymouth,mobile,81.44.176.233,25532 Jacqueline Pine Apt. 146\nSouth Eddiesid...,25532 Jacqueline Pine Apt. 146\nSouth Eddiesid...,0,159,14
5,8cdc7e1c-3376-43c9-abd7-2aaecd0c7560,e18267a3-ab64-4219-8c5b-9a40b45b0f4c,145.75,2024-01-29 15:27:54,credit card,toys & games,4,23,North Sheila,desktop,140.32.231.106,"1041 Jeremy Light Apt. 091\nGarciaton, ID 51416","1041 Jeremy Light Apt. 091\nGarciaton, ID 51416",0,10,15
6,f069bdd8-d4ba-4fac-bbc3-15fa3f0e4879,40eeb902-253d-41d8-bbd3-386c76f8c2e2,36.96,2024-01-29 17:07:10,credit card,toys & games,5,54,Emilyhaven,tablet,24.129.227.90,"96170 Nicholas Flat\nMichelleton, GU 39764","96170 Nicholas Flat\nMichelleton, GU 39764",0,202,17
7,6538d3a9-ab86-4c4d-87de-3720967daabb,9bc42554-3067-4d03-b277-98abd999aacf,187.30,2024-01-29 17:19:42,credit card,electronics,2,49,East Gerald,mobile,42.164.176.126,"5686 Matthew Locks\nLake Kim, TX 30217","5686 Matthew Locks\nLake Kim, TX 30217",0,41,17
8,893b7273-ac5b-4ed4-9f39-caef88d471f5,e82a1c86-7ddb-4fa0-a14d-9d75798026c1,53.98,2024-01-29 17:35:19,bank transfer,home & garden,1,42,West Alexis,mobile,117.9.134.167,"0475 Richard Path Suite 674\nEast Laurahaven, ...","0475 Richard Path Suite 674\nEast Laurahaven, ...",0,256,17
9,1225d735-b3ef-4aa7-b865-b7d975a64aee,90a3e830-0db0-482b-8d66-78cf9ca4e695,265.48,2024-01-29 17:59:39,debit card,home & garden,1,48,North Aliciafurt,mobile,29.247.189.90,"11707 Jackson Forge\nStrongville, MT 34235","11707 Jackson Forge\nStrongville, MT 34235",1,169,17


In [9]:
import os

STREAM_SAMPLE_PATH = "/home/lucas/projects/fraud-bigdata/data/raw/stream_demo_1000.csv"

os.makedirs(os.path.dirname(STREAM_SAMPLE_PATH), exist_ok=True)

stream_demo.to_csv(STREAM_SAMPLE_PATH, index=False)

print("Saved:", STREAM_SAMPLE_PATH)
print("Rows:", len(stream_demo))
print("Frauds:", int(stream_demo[TARGET_COL].sum()))

Saved: /home/lucas/projects/fraud-bigdata/data/raw/stream_demo_1000.csv
Rows: 1000
Frauds: 50


In [11]:
check_stream = pd.read_csv(STREAM_SAMPLE_PATH)

print(check_stream.shape)
print(check_stream[TARGET_COL].value_counts())


(1000, 16)
Is Fraudulent
0    950
1     50
Name: count, dtype: int64


In [12]:
import pandas as pd
import numpy as np

TARGET_COL = "Is Fraudulent"
DATE_COL = "Transaction Date"

def feature_engineering_base(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    df["address_mismatch"] = (
        df["Shipping Address"].astype(str).str.strip()
        != df["Billing Address"].astype(str).str.strip()
    ).astype(int)

    df["is_night"] = (df["Transaction Hour"] <= 5).astype(int)
    df["amount_log"] = np.log1p(df["Transaction Amount"])
    df["amount_per_qty"] = df["Transaction Amount"] / df["Quantity"].replace(0, 1)
    df["account_age_days"] = df["Account Age Days"]
    df["customer_age"] = df["Customer Age"].clip(lower=10)
    df["tx_hour"] = df[DATE_COL].dt.hour
    df["tx_dayofweek"] = df[DATE_COL].dt.dayofweek

    return df

train_feat = feature_engineering_base(raw_train)
test_feat = feature_engineering_base(raw_test)

print("Base features created")
print("Train:", train_feat.shape)
print("Test:", test_feat.shape)

Base features created
Train: (240000, 24)
Test: (60000, 24)


In [13]:
# DROP identifier & leaky columns
DROP_COLS = [
    "Transaction ID", "Customer ID", DATE_COL,
    "IP Address", "Customer Location",
    "Shipping Address", "Billing Address",
    "Customer Age", "Account Age Days",
    "Transaction Hour", "Transaction Amount",
]
DROP_COLS = [c for c in DROP_COLS if c in train_feat.columns]

X_train = train_feat.drop(columns=DROP_COLS + [TARGET_COL])
y_train = train_feat[TARGET_COL]

X_test = test_feat.drop(columns=DROP_COLS + [TARGET_COL])
y_test = test_feat[TARGET_COL]

# Fix customer_age using train median only
age_median = X_train["customer_age"][X_train["customer_age"] >= 18].median()

for df_ in [X_train, X_test]:
    df_["customer_age"] = df_["customer_age"].apply(
        lambda x: age_median if x < 18 else x
    )

# Group statistics from train only
train_tmp = X_train.copy()
train_tmp["__label__"] = y_train.values
global_mean = float(y_train.mean())
smoothing = 20

encoding_maps = {}

for col_name in ["Payment Method", "Product Category", "Device Used"]:
    stats = (
        train_tmp.groupby(col_name)["__label__"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "fraud_rate", "count": "n"})
    )

    stats["smoothed"] = (
        (stats["n"] * stats["fraud_rate"] + smoothing * global_mean)
        / (stats["n"] + smoothing)
    )

    feat_name = col_name.lower().replace(" ", "_") + "_fraud_rate"
    encode_map = stats["smoothed"].to_dict()
    encoding_maps[feat_name] = {str(k): float(v) for k, v in encode_map.items()}

    X_train[feat_name] = X_train[col_name].map(encode_map).fillna(global_mean)
    X_test[feat_name] = X_test[col_name].map(encode_map).fillna(global_mean)

# Category mean stats from train only
cat_amount_mean = train_tmp.groupby("Product Category")["amount_log"].mean().to_dict()
cat_qty_mean = train_tmp.groupby("Product Category")["Quantity"].mean().to_dict()

for df_ in [X_train, X_test]:
    df_["amount_vs_category"] = (
        df_["amount_log"] - df_["Product Category"].map(cat_amount_mean).fillna(0)
    )
    df_["qty_vs_category"] = (
        df_["Quantity"] - df_["Product Category"].map(cat_qty_mean).fillna(0)
    )

# Pattern features using train thresholds
amt_q80 = float(X_train["amount_log"].quantile(0.80))
age_q25_train = float(X_train["account_age_days"].quantile(0.25))

for df_ in [X_train, X_test]:
    df_["is_new_account"] = (df_["account_age_days"] <= age_q25_train).astype(int)
    df_["is_very_new_account"] = (df_["account_age_days"] <= 30).astype(int)
    df_["account_age_clipped"] = df_["account_age_days"].clip(upper=60)
    df_["is_high_amount"] = (df_["amount_log"] >= amt_q80).astype(int)
    df_["new_x_night"] = df_["is_new_account"] * df_["is_night"]
    df_["amount_x_night"] = df_["amount_log"] * df_["is_night"]
    df_["amount_x_very_new"] = df_["amount_log"] * df_["is_very_new_account"]
    df_["mismatch_x_amount"] = df_["address_mismatch"] * df_["amount_log"]
    df_["new_x_highamount"] = df_["is_very_new_account"] * df_["is_high_amount"]
    df_["new_night_highamount"] = (
        df_["is_very_new_account"] * df_["is_night"] * df_["is_high_amount"]
    )

print("Feature params fitted from train only")
print("age_median:", age_median)
print("amt_q80:", amt_q80)
print("age_q25_train:", age_q25_train)
print("Train fraud:", round(y_train.mean() * 100, 4), "%")
print("Test fraud:", round(y_test.mean() * 100, 4), "%")

Feature params fitted from train only
age_median: 35.0
amt_q80: 5.839012423916305
age_q25_train: 87.0
Train fraud: 4.9979 %
Test fraud: 4.9983 %


In [14]:
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler, StringIndexer, OneHotEncoder as SparkOHE
from pyspark.ml.classification import GBTClassifier

train_pd = X_train.copy()
train_pd["label"] = y_train.values

test_pd = X_test.copy()
test_pd["label"] = y_test.values

train_spark = spark.createDataFrame(train_pd)
test_spark = spark.createDataFrame(test_pd)

print("Spark train:", train_spark.count())
print("Spark test:", test_spark.count())

SPARK_NUM = [
    "amount_log", "account_age_clipped", "tx_hour",
    "amount_x_very_new", "amount_vs_category", "qty_vs_category",
    "payment_method_fraud_rate", "product_category_fraud_rate",
    "device_used_fraud_rate",
]

SPARK_BIN = [
    "is_night", "is_very_new_account", "is_high_amount",
    "new_x_night", "new_night_highamount", "new_x_highamount",
]

SPARK_CAT = ["Payment Method", "Product Category", "Device Used"]

SPARK_NUM = [c for c in SPARK_NUM if c in train_pd.columns]
SPARK_BIN = [c for c in SPARK_BIN if c in train_pd.columns]
SPARK_CAT = [c for c in SPARK_CAT if c in train_pd.columns]

# class weight
neg_n = train_spark.filter(col("label") == 0).count()
pos_n = train_spark.filter(col("label") == 1).count()
w_pos = float(neg_n / pos_n)

train_spark_w = train_spark.withColumn(
    "classWeight",
    when(col("label") == 1, w_pos).otherwise(1.0)
)

print("scale_pos_weight / classWeight fraud:", round(w_pos, 4))

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in SPARK_CAT
]

encoders = [
    SparkOHE(inputCol=f"{c}_idx", outputCol=f"{c}_ohe", handleInvalid="keep")
    for c in SPARK_CAT
]

ohe_cols = [f"{c}_ohe" for c in SPARK_CAT]

assembler = VectorAssembler(
    inputCols=SPARK_NUM + SPARK_BIN + ohe_cols,
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = SparkScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

gbt_clf = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="classWeight",
    maxIter=50,
    maxDepth=4,
    stepSize=0.05,
    seed=42
)

gbt_pipeline = SparkPipeline(
    stages=indexers + encoders + [assembler, scaler, gbt_clf]
)

print("Training GBT Spark MLlib...")
fitted_gbt = gbt_pipeline.fit(train_spark_w)
print("GBT training done")

Spark train: 240000


Spark test: 60000


scale_pos_weight / classWeight fraud: 19.0083
Training GBT Spark MLlib...


Traceback (most recent call last):                                              
  File "/opt/spark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/spark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


GBT training done


In [15]:
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    fbeta_score, f1_score, precision_score, recall_score, confusion_matrix,
    classification_report
)
import numpy as np
import pandas as pd

preds = fitted_gbt.transform(test_spark)

eval_pdf = preds.select(
    "label",
    vector_to_array("probability")[1].alias("prob")
).toPandas()

y_true = eval_pdf["label"].values
y_prob = eval_pdf["prob"].values

pr_auc = average_precision_score(y_true, y_prob)
roc_auc = roc_auc_score(y_true, y_prob)

best_t, best_f2 = 0.5, 0

for t in np.arange(0.10, 0.90, 0.01):
    y_pred_t = (y_prob >= t).astype(int)
    f2 = fbeta_score(y_true, y_pred_t, beta=2, zero_division=0)
    if f2 > best_f2:
        best_f2 = f2
        best_t = float(t)

y_pred = (y_prob >= best_t).astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("=" * 60)
print(f"FINAL LOCAL GBT EVALUATION | threshold = {best_t:.2f}")
print("=" * 60)
print(f"PR-AUC    : {pr_auc:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"F2-score  : {best_f2:.4f}")
print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"Precision : {precision_score(y_true, y_pred):.4f}")
print()
print(f"TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}")
print()
print(classification_report(y_true, y_pred, target_names=["Non-Fraud", "Fraud"]))

FINAL LOCAL GBT EVALUATION | threshold = 0.65
PR-AUC    : 0.5786
ROC-AUC   : 0.8454
F2-score  : 0.5339
Recall    : 0.6682
Precision : 0.2959

TN=52,233 | FP=4,768 | FN=995 | TP=2,004

              precision    recall  f1-score   support

   Non-Fraud       0.98      0.92      0.95     57001
       Fraud       0.30      0.67      0.41      2999

    accuracy                           0.90     60000
   macro avg       0.64      0.79      0.68     60000
weighted avg       0.95      0.90      0.92     60000



In [16]:
import os
import json
import subprocess

LOCAL_MODEL_DIR = "/home/lucas/projects/fraud-bigdata/models/gbt_fraud_pipeline"
LOCAL_CONFIG_PATH = "/home/lucas/projects/fraud-bigdata/models/gbt_feature_config.json"

HDFS_MODEL_DIR = "hdfs://localhost:9000/user/lucas/projects/fraud_project/models/gbt_fraud_pipeline"
HDFS_CONFIG_DIR = "/user/lucas/projects/fraud_project/models/config"
HDFS_CONFIG_PATH = f"{HDFS_CONFIG_DIR}/gbt_feature_config.json"

# remove old local model if exists
subprocess.run(["rm", "-rf", LOCAL_MODEL_DIR])

# remove old hdfs model/config if exists
subprocess.run(["hdfs", "dfs", "-rm", "-r", "-f", "/user/lucas/projects/fraud_project/models/gbt_fraud_pipeline"])
subprocess.run(["hdfs", "dfs", "-mkdir", "-p", HDFS_CONFIG_DIR])

# save Spark PipelineModel directly to HDFS
fitted_gbt.write().overwrite().save(HDFS_MODEL_DIR)

feature_config = {
    "target_col": TARGET_COL,
    "date_col": DATE_COL,
    "age_median": float(age_median),
    "global_mean": float(global_mean),
    "smoothing": int(smoothing),
    "amt_q80": float(amt_q80),
    "age_q25_train": float(age_q25_train),
    "best_threshold": float(best_t),
    "encoding_maps": encoding_maps,
    "cat_amount_mean": {str(k): float(v) for k, v in cat_amount_mean.items()},
    "cat_qty_mean": {str(k): float(v) for k, v in cat_qty_mean.items()},
    "spark_num": SPARK_NUM,
    "spark_bin": SPARK_BIN,
    "spark_cat": SPARK_CAT
}

os.makedirs(os.path.dirname(LOCAL_CONFIG_PATH), exist_ok=True)

with open(LOCAL_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_config, f, indent=2, ensure_ascii=False)

# put config to HDFS
subprocess.run(["hdfs", "dfs", "-put", "-f", LOCAL_CONFIG_PATH, HDFS_CONFIG_PATH])

print("Saved Spark PipelineModel to:", HDFS_MODEL_DIR)
print("Saved config local:", LOCAL_CONFIG_PATH)
print("Saved config HDFS:", HDFS_CONFIG_PATH)

Saved Spark PipelineModel to: hdfs://localhost:9000/user/lucas/projects/fraud_project/models/gbt_fraud_pipeline
Saved config local: /home/lucas/projects/fraud-bigdata/models/gbt_feature_config.json
Saved config HDFS: /user/lucas/projects/fraud_project/models/config/gbt_feature_config.json


In [17]:
import subprocess

print("=== HDFS models ===")
subprocess.run(["hdfs", "dfs", "-ls", "-R", "/user/lucas/projects/fraud_project/models"])

print("\n=== Local models folder ===")
subprocess.run(["ls", "-lh", "/home/lucas/projects/fraud-bigdata/models"])

=== HDFS models ===
drwxr-xr-x   - lucas supergroup          0 2026-06-11 12:05 /user/lucas/projects/fraud_project/models/config
-rw-r--r--   1 lucas supergroup       1824 2026-06-11 12:05 /user/lucas/projects/fraud_project/models/config/gbt_feature_config.json
drwxr-xr-x   - lucas supergroup          0 2026-06-11 12:04 /user/lucas/projects/fraud_project/models/gbt_fraud_pipeline
drwxr-xr-x   - lucas supergroup          0 2026-06-11 12:04 /user/lucas/projects/fraud_project/models/gbt_fraud_pipeline/metadata
-rw-r--r--   1 lucas supergroup          0 2026-06-11 12:04 /user/lucas/projects/fraud_project/models/gbt_fraud_pipeline/metadata/_SUCCESS
-rw-r--r--   1 lucas supergroup        442 2026-06-11 12:04 /user/lucas/projects/fraud_project/models/gbt_fraud_pipeline/metadata/part-00000-c3cf6561-e057-47bb-beac-a344cd9e270f-c000.txt
drwxr-xr-x   - lucas supergroup          0 2026-06-11 12:05 /user/lucas/projects/fraud_project/models/gbt_fraud_pipeline/stages
drwxr-xr-x   - lucas supergroup  

CompletedProcess(args=['ls', '-lh', '/home/lucas/projects/fraud-bigdata/models'], returncode=0)

In [18]:
import json
import pandas as pd
import numpy as np

from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, when, round as spark_round

MODEL_PATH = "hdfs://localhost:9000/user/lucas/projects/fraud_project/models/gbt_fraud_pipeline"
LOCAL_CONFIG_PATH = "/home/lucas/projects/fraud-bigdata/models/gbt_feature_config.json"

loaded_model = PipelineModel.load(MODEL_PATH)

with open(LOCAL_CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

threshold = float(config["best_threshold"])

print("Loaded model:", MODEL_PATH)
print("Loaded config:", LOCAL_CONFIG_PATH)
print("Threshold:", threshold)
print("SPARK_NUM:", config["spark_num"])
print("SPARK_BIN:", config["spark_bin"])
print("SPARK_CAT:", config["spark_cat"])

Loaded model: hdfs://localhost:9000/user/lucas/projects/fraud_project/models/gbt_fraud_pipeline
Loaded config: /home/lucas/projects/fraud-bigdata/models/gbt_feature_config.json
Threshold: 0.6499999999999997
SPARK_NUM: ['amount_log', 'account_age_clipped', 'tx_hour', 'amount_x_very_new', 'amount_vs_category', 'qty_vs_category', 'payment_method_fraud_rate', 'product_category_fraud_rate', 'device_used_fraud_rate']
SPARK_BIN: ['is_night', 'is_very_new_account', 'is_high_amount', 'new_x_night', 'new_night_highamount', 'new_x_highamount']
SPARK_CAT: ['Payment Method', 'Product Category', 'Device Used']


In [20]:
STREAM_SAMPLE_PATH = "/home/lucas/projects/fraud-bigdata/data/raw/stream_demo_1000.csv"

stream_pd_raw = pd.read_csv(STREAM_SAMPLE_PATH)
stream_pd_raw[DATE_COL] = pd.to_datetime(stream_pd_raw[DATE_COL], errors="coerce")

print("Stream sample:", stream_pd_raw.shape)
print(stream_pd_raw[TARGET_COL].value_counts())
print("Fraud rate:", round(stream_pd_raw[TARGET_COL].mean() * 100, 4), "%")


Stream sample: (1000, 16)
Is Fraudulent
0    950
1     50
Name: count, dtype: int64
Fraud rate: 5.0 %


In [21]:
def build_stream_features(raw_data: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = raw_data.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

    # Base row-level features
    df["address_mismatch"] = (
        df["Shipping Address"].astype(str).str.strip()
        != df["Billing Address"].astype(str).str.strip()
    ).astype(int)

    df["is_night"] = (df["Transaction Hour"] <= 5).astype(int)
    df["amount_log"] = np.log1p(df["Transaction Amount"])
    df["amount_per_qty"] = df["Transaction Amount"] / df["Quantity"].replace(0, 1)
    df["account_age_days"] = df["Account Age Days"]
    df["customer_age"] = df["Customer Age"].clip(lower=10)
    df["tx_hour"] = df[DATE_COL].dt.hour
    df["tx_dayofweek"] = df[DATE_COL].dt.dayofweek

    # Drop identifier/leaky/raw columns, same as training
    drop_cols = [
        "Transaction ID", "Customer ID", DATE_COL,
        "IP Address", "Customer Location",
        "Shipping Address", "Billing Address",
        "Customer Age", "Account Age Days",
        "Transaction Hour", "Transaction Amount",
    ]
    drop_cols = [c for c in drop_cols if c in df.columns]

    y = df[TARGET_COL].copy() if TARGET_COL in df.columns else None
    X = df.drop(columns=drop_cols + ([TARGET_COL] if TARGET_COL in df.columns else []))

    # Fix customer_age using training median
    age_median = float(config["age_median"])
    X["customer_age"] = X["customer_age"].apply(lambda x: age_median if x < 18 else x)

    # Target encoding maps from training only
    global_mean = float(config["global_mean"])

    for feat_name, mapping in config["encoding_maps"].items():
        if feat_name == "payment_method_fraud_rate":
            source_col = "Payment Method"
        elif feat_name == "product_category_fraud_rate":
            source_col = "Product Category"
        elif feat_name == "device_used_fraud_rate":
            source_col = "Device Used"
        else:
            continue

        X[feat_name] = X[source_col].astype(str).map(mapping).fillna(global_mean)

    # Category mean stats
    cat_amount_mean = config["cat_amount_mean"]
    cat_qty_mean = config["cat_qty_mean"]

    X["amount_vs_category"] = (
        X["amount_log"] - X["Product Category"].astype(str).map(cat_amount_mean).fillna(0)
    )
    X["qty_vs_category"] = (
        X["Quantity"] - X["Product Category"].astype(str).map(cat_qty_mean).fillna(0)
    )

    # Pattern features
    amt_q80 = float(config["amt_q80"])
    age_q25_train = float(config["age_q25_train"])

    X["is_new_account"] = (X["account_age_days"] <= age_q25_train).astype(int)
    X["is_very_new_account"] = (X["account_age_days"] <= 30).astype(int)
    X["account_age_clipped"] = X["account_age_days"].clip(upper=60)
    X["is_high_amount"] = (X["amount_log"] >= amt_q80).astype(int)
    X["new_x_night"] = X["is_new_account"] * X["is_night"]
    X["amount_x_night"] = X["amount_log"] * X["is_night"]
    X["amount_x_very_new"] = X["amount_log"] * X["is_very_new_account"]
    X["mismatch_x_amount"] = X["address_mismatch"] * X["amount_log"]
    X["new_x_highamount"] = X["is_very_new_account"] * X["is_high_amount"]
    X["new_night_highamount"] = (
        X["is_very_new_account"] * X["is_night"] * X["is_high_amount"]
    )

    # Attach useful identifiers for display
    X["Transaction ID"] = raw_data["Transaction ID"].values
    X["Transaction Amount"] = raw_data["Transaction Amount"].values
    X["Transaction Date"] = raw_data["Transaction Date"].values
    X["Account Age Days"] = raw_data["Account Age Days"].values
    X["Transaction Hour"] = raw_data["Transaction Hour"].values

    if y is not None:
        X["label"] = y.values

    return X

stream_features_pd = build_stream_features(stream_pd_raw, config)

print("Stream features shape:", stream_features_pd.shape)
print("Columns:", stream_features_pd.columns.tolist())
stream_features_pd.head()

Stream features shape: (1000, 33)
Columns: ['Payment Method', 'Product Category', 'Quantity', 'Device Used', 'address_mismatch', 'is_night', 'amount_log', 'amount_per_qty', 'account_age_days', 'customer_age', 'tx_hour', 'tx_dayofweek', 'payment_method_fraud_rate', 'product_category_fraud_rate', 'device_used_fraud_rate', 'amount_vs_category', 'qty_vs_category', 'is_new_account', 'is_very_new_account', 'account_age_clipped', 'is_high_amount', 'new_x_night', 'amount_x_night', 'amount_x_very_new', 'mismatch_x_amount', 'new_x_highamount', 'new_night_highamount', 'Transaction ID', 'Transaction Amount', 'Transaction Date', 'Account Age Days', 'Transaction Hour', 'label']


,Payment Method,Product Category,Quantity,Device Used,address_mismatch,is_night,amount_log,amount_per_qty,account_age_days,customer_age,...,amount_x_very_new,mismatch_x_amount,new_x_highamount,new_night_highamount,Transaction ID,Transaction Amount,Transaction Date,Account Age Days,Transaction Hour,label
0,PayPal,clothing,4,desktop,0,0,6.551794,174.875000,343,18.0,...,0.0,0.0,0,0,06f11434-ac9e-4ef2-9663-c1947b37f8d6,699.50,2024-01-29 12:01:33,343,12,1
1,debit card,toys & games,2,desktop,0,0,4.907938,67.180000,199,39.0,...,0.0,0.0,0,0,320449d0-51fc-4dee-8a79-f36683ca52c4,134.36,2024-01-29 12:29:06,199,12,0
2,credit card,health & beauty,3,mobile,0,0,6.126826,152.326667,141,39.0,...,0.0,0.0,0,0,4b590d1a-73c0-4eed-9278-df32700ec333,456.98,2024-01-29 12:51:03,141,12,0
3,debit card,home & garden,2,desktop,0,0,4.720729,55.625000,232,28.0,...,0.0,0.0,0,0,3fb06b6f-4e42-462a-a577-c1884071929f,111.25,2024-01-29 13:01:49,232,13,0
4,credit card,electronics,3,mobile,0,0,4.800572,40.193333,159,44.0,...,0.0,0.0,0,0,be8376e6-fa95-43a8-a922-cce11fefe19d,120.58,2024-01-29 14:42:59,159,14,0


In [22]:
stream_spark = spark.createDataFrame(stream_features_pd)

pred_stream = loaded_model.transform(stream_spark)

pred_result = pred_stream.select(
    "Transaction ID",
    "Transaction Amount",
    "Transaction Date",
    "Account Age Days",
    "Transaction Hour",
    "Payment Method",
    "Product Category",
    "Device Used",
    "label",
    vector_to_array("probability")[1].alias("fraud_probability"),
    "prediction"
).withColumn(
    "predicted_fraud",
    when(col("fraud_probability") >= threshold, 1).otherwise(0)
).withColumn(
    "risk_tier",
    when(col("fraud_probability") >= threshold, "High Risk")
    .when(col("fraud_probability") >= threshold * 0.7, "Medium Risk")
    .otherwise("Low Risk")
)

pred_result.createOrReplaceTempView("stream_static_predictions")

pred_result.show(20, truncate=False)

+------------------------------------+------------------+-------------------+----------------+----------------+--------------+----------------+-----------+-----+-------------------+----------+---------------+---------+
|Transaction ID                      |Transaction Amount|Transaction Date   |Account Age Days|Transaction Hour|Payment Method|Product Category|Device Used|label|fraud_probability  |prediction|predicted_fraud|risk_tier|
+------------------------------------+------------------+-------------------+----------------+----------------+--------------+----------------+-----------+-----+-------------------+----------+---------------+---------+
|06f11434-ac9e-4ef2-9663-c1947b37f8d6|699.5             |2024-01-29 12:01:33|343             |12              |PayPal        |clothing        |desktop    |1    |0.3853043521864886 |0.0       |0              |Low Risk |
|320449d0-51fc-4dee-8a79-f36683ca52c4|134.36            |2024-01-29 12:29:06|199             |12              |debit card   

In [23]:
summary_stream = spark.sql("""
SELECT
    COUNT(*) AS total_transactions,
    SUM(label) AS actual_frauds,
    SUM(predicted_fraud) AS predicted_frauds,
    SUM(CASE WHEN label = 1 AND predicted_fraud = 1 THEN 1 ELSE 0 END) AS caught_frauds,
    SUM(CASE WHEN label = 1 AND predicted_fraud = 0 THEN 1 ELSE 0 END) AS missed_frauds,
    SUM(CASE WHEN label = 0 AND predicted_fraud = 1 THEN 1 ELSE 0 END) AS false_alarms,
    ROUND(AVG(fraud_probability), 4) AS avg_fraud_probability
FROM stream_static_predictions
""")

summary_stream.show(truncate=False)

risk_summary = spark.sql("""
SELECT
    risk_tier,
    COUNT(*) AS transactions,
    SUM(label) AS actual_frauds,
    ROUND(AVG(fraud_probability), 4) AS avg_fraud_probability
FROM stream_static_predictions
GROUP BY risk_tier
ORDER BY avg_fraud_probability DESC
""")

risk_summary.show(truncate=False)

+------------------+-------------+----------------+-------------+-------------+------------+---------------------+
|total_transactions|actual_frauds|predicted_frauds|caught_frauds|missed_frauds|false_alarms|avg_fraud_probability|
+------------------+-------------+----------------+-------------+-------------+------------+---------------------+
|1000              |50           |102             |31           |19           |71          |0.3145               |
+------------------+-------------+----------------+-------------+-------------+------------+---------------------+



+-----------+------------+-------------+---------------------+
|risk_tier  |transactions|actual_frauds|avg_fraud_probability|
+-----------+------------+-------------+---------------------+
|High Risk  |102         |31           |0.7947               |
|Medium Risk|23          |3            |0.5899               |
|Low Risk   |875         |16           |0.2513               |
+-----------+------------+-------------+---------------------+



In [24]:
spark.sql("""
SELECT
    `Transaction ID`,
    ROUND(`Transaction Amount`, 2) AS amount,
    `Transaction Hour` AS hour,
    `Account Age Days` AS account_age_days,
    `Payment Method`,
    `Product Category`,
    `Device Used`,
    label AS actual_fraud,
    ROUND(fraud_probability, 4) AS fraud_probability,
    risk_tier
FROM stream_static_predictions
ORDER BY fraud_probability DESC
LIMIT 20
""").show(20, truncate=False)

+------------------------------------+-------+----+----------------+--------------+----------------+-----------+------------+-----------------+---------+
|Transaction ID                      |amount |hour|account_age_days|Payment Method|Product Category|Device Used|actual_fraud|fraud_probability|risk_tier|
+------------------------------------+-------+----+----------------+--------------+----------------+-----------+------------+-----------------+---------+
|c04f8a79-9ff5-4394-838a-e8e8a8368ed7|1986.2 |5   |258             |debit card    |toys & games    |desktop    |1           |0.9666           |High Risk|
|b5fa78df-080c-4cc7-a709-a4e47464a57c|497.9  |3   |357             |debit card    |clothing        |tablet     |1           |0.9629           |High Risk|
|fff36e85-e673-4d8f-b078-1741ef702c24|118.24 |3   |56              |credit card   |clothing        |desktop    |1           |0.9629           |High Risk|
|c33eff11-1dd0-4bd2-8de8-e2ffa1cb9876|219.6  |5   |34              |debit ca

## STATIC STREAMING TEST

In [ ]:
from pyspark.sql.functions import col

SHOWCASE_PATH = "/home/lucas/projects/fraud-bigdata/data/raw/stream_showcase_30.csv"

top_ids_pdf = (
    pred_result
    .orderBy(col("fraud_probability").desc())
    .select("Transaction ID")
    .limit(30)
    .toPandas()
)

top_ids = top_ids_pdf["Transaction ID"].tolist()
rank_map = {tid: i for i, tid in enumerate(top_ids)}

showcase_pd = stream_pd_raw[stream_pd_raw["Transaction ID"].isin(top_ids)].copy()
showcase_pd["__rank"] = showcase_pd["Transaction ID"].map(rank_map)
showcase_pd = showcase_pd.sort_values("__rank").drop(columns="__rank")

showcase_pd.to_csv(SHOWCASE_PATH, index=False)

print("Saved:", SHOWCASE_PATH)
print("Rows:", len(showcase_pd))
print(showcase_pd["Is Fraudulent"].value_counts())
showcase_pd[[
    "Transaction ID", "Transaction Amount", "Transaction Hour",
    "Account Age Days", "Payment Method", "Product Category",
    "Device Used", "Is Fraudulent"
]].head(10)

Saved: /home/lucas/projects/fraud-bigdata/data/raw/stream_showcase_30.csv
Rows: 30
Is Fraudulent
1    24
0     6
Name: count, dtype: int64


,Transaction ID,Transaction Amount,Transaction Hour,Account Age Days,Payment Method,Product Category,Device Used,Is Fraudulent
998,c04f8a79-9ff5-4394-838a-e8e8a8368ed7,1986.20,5,258,debit card,toys & games,desktop,1
74,fff36e85-e673-4d8f-b078-1741ef702c24,118.24,3,56,credit card,clothing,desktop,1
847,f58d8268-1c10-4bef-a4d7-140565c536ef,143.20,1,5,PayPal,clothing,mobile,1
553,c33eff11-1dd0-4bd2-8de8-e2ffa1cb9876,219.60,5,34,debit card,clothing,desktop,1
871,d0afdc3d-e5f9-4a0a-975d-50988a1f229b,192.77,5,7,bank transfer,health & beauty,tablet,1
82,020ae1f2-3bc0-44d1-af99-7038d2ceb576,474.50,2,15,debit card,clothing,mobile,1
873,a948d2df-c8c2-402a-b844-2395f7f6b769,182.42,3,196,debit card,toys & games,tablet,1
606,2cbb55d4-f6b4-4637-b053-3d5a19452153,199.99,1,11,bank transfer,home & garden,mobile,1
936,49129c9c-60c6-4642-9d27-080cabd080aa,11.07,1,249,credit card,home & garden,mobile,1
125,87a28ab4-c9a8-4cc1-8534-7524f1a97bda,294.15,5,30,credit card,toys & games,tablet,1


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 49860)
Traceback (most recent call last):
  File "/usr/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/home/lucas/projects/fraud-bigdata/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/home/lucas/projects/fraud-bigdata/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 272, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/home/lucas/projects/fraud